# mfs-agent-security: benchmark run on Kaggle

**Before running:** in the right panel, set **Accelerator: GPU T4 x2** and turn **Internet on** (needs a phone-verified Kaggle account).

Then run the cells in order (Run All works). Expect roughly 20 to 40 minutes, mostly installing vLLM and loading the model.

In [ ]:
!nvidia-smi

## 1. Get the project and run its tests

In [ ]:
!git clone -q https://github.com/s4m404/mfs-agent-security.git
%cd mfs-agent-security
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Install vLLM and start the model server
vLLM has reliable tool calling for Qwen models (the hermes parser). The second cell waits until the server is ready.

In [ ]:
!pip install -q vllm

In [ ]:
import subprocess, time, urllib.request

MODEL = "Qwen/Qwen2.5-7B-Instruct"

server = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--dtype", "half",                 # T4 GPUs do not support bfloat16
     "--tensor-parallel-size", "2",     # split the model across both T4s
     "--max-model-len", "8192",
     "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
     "--port", "8000"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

for i in range(90):  # wait up to 15 minutes
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"Server ready after about {i * 10} seconds")
        break
    except Exception:
        if server.poll() is not None:
            print("Server stopped. Last lines of vllm.log:")
            print(open("vllm.log").read()[-3000:])
            break
        time.sleep(10)
else:
    print("Still not ready. Last lines of vllm.log:")
    print(open("vllm.log").read()[-3000:])

## 3. Sanity check: does the model make real tool calls?
You should see a `check_balance` tool call. If `tool_calls` is `None`, the benchmark results will not be meaningful, so stop and send Claude this output.

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
resp = client.chat.completions.create(
    model=MODEL, temperature=0,
    messages=[{"role": "user", "content": "What is my wallet balance?"}],
    tools=[{"type": "function", "function": {
        "name": "check_balance", "description": "Return the user's balance.",
        "parameters": {"type": "object", "properties": {}}}}])
msg = resp.choices[0].message
print("tool_calls:", msg.tool_calls)
print("content:", msg.content)

## 4. Run the benchmark
Three runs: no defence, provenance defence, and no defence with the plain system prompt.

In [ ]:
for defence, prompt in [("none", "guarded"), ("provenance", "guarded"), ("none", "plain")]:
    print(f"\n===== defence={defence} prompt={prompt} =====")
    !python scripts/run_bench.py --model {MODEL} --base-url http://localhost:8000/v1 --defence {defence} --prompt {prompt} 2>&1 | grep -v "HTTP Request"


## 5. Diagnostics
Tool calls per run. Zero tool calls everywhere means something is wrong.

In [ ]:
import json, glob, collections

for path in sorted(glob.glob("results/*/traces.jsonl")):
    calls, cases = collections.Counter(), set()
    for line in open(path):
        e = json.loads(line)
        cases.add(e["case_id"])
        if e["type"] == "tool_call":
            calls[e["case_id"]] += 1
    zero = [c for c in sorted(cases) if calls[c] == 0]
    print(path.split("/")[1])
    print(f"  cases with zero tool calls: {len(zero)} of {len(cases)} {zero if zero else ''}")

In [ ]:
# Step by step trace of the simplest task in the no-defence run
run = glob.glob("results/*none__guarded/traces.jsonl")[0]
for line in open(run):
    e = json.loads(line)
    if e["case_id"] == "benign-bill-en":
        print(e["type"], e.get("tool"), e.get("args"), (e.get("result") or e.get("text") or "")[:300])

## 6. Save the results
This creates `results.zip` in the Output panel on the right. Send Claude the printed summaries and diagnostics, or the zip.

In [ ]:
!zip -qr /kaggle/working/results.zip results
for p in sorted(glob.glob("results/*/summary.md")):
    print(open(p).read())